In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("TMDB_all_movies.csv")
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.106,371.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,...,suomi,"Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9745.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.300,435.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,...,"svenska, suomi, English","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8624.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.900,2821.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,English,"David Proval, Tamlyn Tomita, Paul Skemp, Lana ...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Andrzej Sekula, Rodrigo García, Phil Parmet, G...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Alexandre Rockwell, Lawrenc...",Combustible Edison,6.7,116866.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,6,Judgment Night,6.500,370.0,Released,1993-10-15,12136938.0,109.0,21000000.0,tt0107286,...,English,"Jeremy Piven, Lydell M. Cheshier, Michael DeLo...",Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Lloyd Segan, Gene Levy, Marilyn Vance",Alan Silvestri,6.6,21041.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,8,Life in Loops (A Megacities RMX),7.200,30.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,Wolfgang Thaler,"Timo Novotny, Michael Glawogger","Timo Novotny, Ulrich Gehmacher",NaN,8.1,285.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg


In [4]:
df.shape

(1187522, 28)

In [5]:
df.columns.tolist()

['id',
 'title',
 'vote_average',
 'vote_count',
 'status',
 'release_date',
 'revenue',
 'runtime',
 'budget',
 'imdb_id',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'tagline',
 'genres',
 'production_companies',
 'production_countries',
 'spoken_languages',
 'cast',
 'director',
 'director_of_photography',
 'writers',
 'producers',
 'music_composer',
 'imdb_rating',
 'imdb_votes',
 'poster_path']

In [6]:
df.isnull().sum()

id                               0
title                            9
vote_average                     0
vote_count                       0
status                           0
release_date                124805
revenue                          0
runtime                          0
budget                           0
imdb_id                     523002
original_language                0
original_title                   8
overview                    181545
popularity                       0
tagline                    1007472
genres                      319856
production_companies        615501
production_countries        440538
spoken_languages            432621
cast                        377250
director                    191638
director_of_photography     868033
writers                     586598
producers                   781869
music_composer             1052803
imdb_rating                 720005
imdb_votes                  720005
poster_path                 290513
dtype: int64

In [7]:
# 3. Drop exact duplicate rows
df = df.drop_duplicates()
print("After dropping duplicates:", df.shape)


After dropping duplicates: (1187522, 28)


In [8]:
# 4. Drop columns with too many missing values
cols_to_drop = [
    "director_of_photography",
    "writers",
    "producers",
    "music_composer",
    "imdb_rating",
    "imdb_votes",
    "poster_path"
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


In [9]:
# 5. Handle core text columns: title / original_title / overview
# Drop rows with missing title or overview because they are essential for a recommender
df = df[df["title"].notna()]
df = df[df["overview"].notna()]

In [10]:
# If original_title is missing but title exists, fill it
df["original_title"] = df["original_title"].fillna(df["title"])

In [11]:
# 6. Convert release_date to datetime and create release_year
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

# Optional: drop rows with completely unknown release date
df = df[df["release_year"].notna()]

In [12]:
# 7. Convert numeric columns safely
num_cols = ["vote_average", "vote_count", "runtime", "budget", "revenue", "popularity"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Optional: remove movies with zero or NaN runtime (bad data)
df = df[df["runtime"].notna()]
df = df[df["runtime"] > 0]

In [13]:
# 8. Fill missing values in remaining text columns with empty string
text_cols = [
    "status",
    "original_language",
    "overview",
    "tagline",
    "genres",
    "production_companies",
    "production_countries",
    "spoken_languages",
    "cast",
    "director"
]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("")


In [14]:
df.shape

(822873, 22)

In [15]:
df.isnull().sum()

id                           0
title                        0
vote_average                 0
vote_count                   0
status                       0
release_date                 0
revenue                      0
runtime                      0
budget                       0
imdb_id                 287843
original_language            0
original_title               0
overview                     0
popularity                   0
tagline                      0
genres                       0
production_companies         0
production_countries         0
spoken_languages             0
cast                         0
director                     0
release_year                 0
dtype: int64

In [18]:
df = df.drop(columns=["imdb_id"])

In [20]:
#  Create a combined text feature for content-based recommendation
df["tags"] = (
    df["genres"] + " " +
    df["cast"] + " " +
    df["director"] + " " +
    df["overview"] + " " +
    df["tagline"]
).str.lower()

In [22]:
final_cols = [
    "id", "title", "status", "original_title", "release_date", "release_year",
    "vote_average", "vote_count", "runtime", "budget", "revenue",
    "original_language", "genres", "cast", "director", "overview",
    "tagline", "popularity", "tags"
]
df_final = df[final_cols]


In [24]:
# 12. Save cleaned dataset
df_final.to_csv("tmdb_cleaned.csv", index=False)
df_final.head()

,id,title,status,original_title,release_date,release_year,vote_average,vote_count,runtime,budget,revenue,original_language,genres,cast,director,overview,tagline,popularity,tags
0,2,Ariel,Released,Ariel,1988-10-21,1988.0,7.106,371.0,73.0,0.0,0.0,fi,"Comedy, Drama, Romance, Crime","Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki,A Finnish man goes to the city to find a job a...,,1.6384,"comedy, drama, romance, crime kari helaseppä, ..."
1,3,Shadows in Paradise,Released,Varjoja paratiisissa,1986-10-17,1986.0,7.300,435.0,74.0,0.0,0.0,fi,"Comedy, Drama, Romance","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,"Nikander, a rubbish collector and would-be ent...",,2.7136,"comedy, drama, romance ari korhonen, mari rant..."
2,5,Four Rooms,Released,Four Rooms,1995-12-09,1995.0,5.900,2821.0,98.0,4000000.0,4257354.0,en,Comedy,"David Proval, Tamlyn Tomita, Paul Skemp, Lana ...","Quentin Tarantino, Robert Rodriguez, Alexandre...",It's Ted the Bellhop's first night on the job....,Twelve outrageous guests. Four scandalous requ...,4.0661,"comedy david proval, tamlyn tomita, paul skemp..."
3,6,Judgment Night,Released,Judgment Night,1993-10-15,1993.0,6.500,370.0,109.0,21000000.0,12136938.0,en,"Action, Crime, Thriller","Jeremy Piven, Lydell M. Cheshier, Michael DeLo...",Stephen Hopkins,"Four young friends, while taking a shortcut en...",Don't move. Don't whisper. Don't even breathe.,2.1596,"action, crime, thriller jeremy piven, lydell m..."
4,8,Life in Loops (A Megacities RMX),Released,Life in Loops (A Megacities RMX),2006-01-01,2006.0,7.200,30.0,80.0,42000.0,0.0,en,Documentary,,Timo Novotny,Timo Novotny labels his new project an experim...,A Megacities remix.,2.2585,documentary timo novotny timo novotny labels ...
